In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path
from glob import glob

root = Path.cwd().resolve()
dataset_candidates = [
    root / "covid-chestxray-dataset",
    root.parent / "covid-chestxray-dataset",
    root / ".." / "covid-chestxray-dataset",
]
path_to_covid_dataset = str(next((p for p in dataset_candidates if p.exists()), (root.parent / "covid-chestxray-dataset").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
import torch.nn.functional as F
import pandas as pd
import sklearn
import utils


In [3]:
import torchxrayvision as xrv

In [4]:
nn_model = xrv.models.DenseNet(weights="all")#.cuda()
nn_model.op_threshs = None # to prevent sigmoid

In [5]:
import torchvision, torchvision.transforms
transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])

In [6]:
# data is at `git checkout 8e7e5900032e0eacbfee0e604561ea3a258e2b2a`
# https://github.com/ieee8023/covid-chestxray-dataset
path_to_covid_dataset = str(Path(path_to_covid_dataset).resolve())


In [7]:
dmerge = xrv.datasets.COVID19_Dataset(imgpath=path_to_covid_dataset + "/images",
                                      csvpath=path_to_covid_dataset + "/metadata.csv",
                                      transform=transform, views=["PA", "AP"])

In [8]:
dmerge.csv = dmerge.csv.reset_index()

In [9]:
dmerge

{'Aspergillosis': {np.float32(0.0): 534, np.float32(1.0): 1},
 'Aspiration': {np.float32(0.0): 534, np.float32(1.0): 1},
 'Bacterial': {np.float32(0.0): 487, np.float32(1.0): 48},
 'COVID-19': {np.float32(0.0): 193, np.float32(1.0): 342},
 'Chlamydophila': {np.float32(0.0): 534, np.float32(1.0): 1},
 'Fungal': {np.float32(0.0): 512, np.float32(1.0): 23},
 'H1N1': {np.float32(0.0): 534, np.float32(1.0): 1},
 'Herpes ': {np.float32(0.0): 532, np.float32(1.0): 3},
 'Influenza': {np.float32(0.0): 531, np.float32(1.0): 4},
 'Klebsiella': {np.float32(0.0): 526, np.float32(1.0): 9},
 'Legionella': {np.float32(0.0): 526, np.float32(1.0): 9},
 'Lipoid': {np.float32(0.0): 527, np.float32(1.0): 8},
 'MERS-CoV': {np.float32(0.0): 527, np.float32(1.0): 8},
 'MRSA': {np.float32(0.0): 534, np.float32(1.0): 1},
 'Mycoplasma': {np.float32(0.0): 530, np.float32(1.0): 5},
 'No Finding': {np.float32(0.0): 520, np.float32(1.0): 15},
 'Nocardia': {np.float32(0.0): 531, np.float32(1.0): 4},
 'Pneumocystis': 

COVID19_Dataset num_samples=535 views=['PA', 'AP'] data_aug=None

In [10]:
dmerge.csv = dmerge.csv.join(pd.DataFrame(dmerge.labels, columns=["gt_" + path for path in dmerge.pathologies]))

In [11]:
scores = pd.read_csv(path_to_covid_dataset + "/annotations/covid-severity-scores.csv", skiprows=5)
scores = scores.set_index("filename")

In [12]:
scores.head()

,geographic_mean,opacity_mean
filename,,
01E392EE-69F9-4E33-BFCE-E5C968654078.jpeg,6.0,4.0
03BF7561-A9BA-4C3C-B8A0-D3E585F73F3C.jpeg,2.7,2.0
1-s2.0-S0140673620303706-fx1_lrg.jpg,2.0,2.0
1-s2.0-S1684118220300608-main.pdf-001.jpg,3.7,3.0
1-s2.0-S1684118220300608-main.pdf-002.jpg,1.7,1.7


In [13]:
dmerge.csv = dmerge.csv.set_index("filename").join(scores, on="filename").reset_index()

In [14]:
dmerge.csv.geographic_mean[~dmerge.csv.geographic_mean.isna()].describe()

count    90.000000
mean      3.280000
std       2.332631
min       0.000000
25%       1.300000
50%       3.000000
75%       4.700000
max       8.000000
Name: geographic_mean, dtype: float64

In [15]:
dl = torch.utils.data.DataLoader(dmerge, batch_size=32, num_workers=2, pin_memory=True)
d_feats = []
d_preds = []
d_pixels = []
with torch.no_grad():
    for i, batch in enumerate(dl):
        imgs = batch["img"]
        out = nn_model.features(imgs)
        out = F.relu(out, inplace=True)
        out = F.adaptive_avg_pool2d(out, (1, 1))
        pred = nn_model.classifier(out.view(out.size(0), -1))
        d_pixels.append(imgs)
        d_feats.append(out.detach().cpu().numpy())
        d_preds.append(pred.detach().cpu().numpy())
        if i % 10 == 0:
            print(i)
d_feats = np.concatenate(d_feats).reshape(-1, 1024)
d_preds = np.concatenate(d_preds)
d_pixels = np.concatenate(d_pixels)


0


10


In [16]:
d_feats.shape

(535, 1024)

In [17]:
dmerge.csv = dmerge.csv.join(pd.DataFrame(d_preds, columns=xrv.datasets.default_pathologies))

In [18]:
dmerge.csv["country"] = dmerge.csv.location.str.split(",").str[-1].str.strip()

In [19]:
continents = pd.read_csv("continents.csv.zip")
continents = continents[["name", "region", "sub-region"]]

In [20]:
dmerge.csv = dmerge.csv.join(continents.set_index("name"), on=["country"])

In [21]:
# should be empty (it not then change the country name)
dmerge.csv [(~dmerge.csv.country.isna() & dmerge.csv.region.isna())].country

Series([], Name: country, dtype: object)

In [22]:
d_preds.shape

(535, 18)

In [23]:
d_feats.shape

(535, 1024)

In [24]:
dmerge.csv.country = dmerge.csv.country.fillna("Unknown")
dmerge.csv.region = dmerge.csv.region.fillna("Unknown")
dmerge.csv["sub-region"] = dmerge.csv["sub-region"].fillna("Unknown")

In [25]:
dmerge.csv["Frontal"] = dmerge.csv.view.isin(["AP", "PA", "AP Supine"])
dmerge.csv["Lateral"] = dmerge.csv.view.isin(["L"])
dmerge.csv["SurvivalY"] = dmerge.csv.survival.isin(["Y"])
dmerge.csv["SurvivalN"] = dmerge.csv.survival.isin(["N"])
dmerge.csv["Went ICUY"] = dmerge.csv.went_icu.isin(["Y"])
dmerge.csv["Went ICUN"] = dmerge.csv.went_icu.isin(["N"])
dmerge.csv["IntubationY"] = dmerge.csv.intubated.isin(["Y"])
dmerge.csv["IntubationN"] = dmerge.csv.intubated.isin(["N"])
dmerge.csv["COVID-19"] = dmerge.csv["finding"].str.contains("COVID-19")
dmerge.csv["RT-PCR Positive"] = dmerge.csv["RT_PCR_positive"].isin(["Y"])
dmerge.csv["Male Ratio"] = dmerge.csv.sex == "M"

In [26]:
dmerge.csv.groupby("patientid").first()[["age","Male Ratio"]].mean()

age           53.957597
Male Ratio     0.556818
dtype: float64

In [27]:
r = dmerge.csv.groupby("patientid").first().groupby(["region", "sub-region"]).agg({
                                                         'filename':["count"],
                                                         'age':['mean'],
                                                         'SurvivalY':['sum'],
                                                         'SurvivalN':['sum'],
                                                         'Went ICUY':['sum'],
                                                         'Went ICUN':['sum'],
                                                         'IntubationY':['sum'],
                                                         'IntubationN':['sum'],
                                                         'COVID-19':['sum'],
                                                         'RT-PCR Positive':["sum"],
                                                         'Male Ratio':['mean'],
                                                        }).round(2)
def merge(df,a):
    df[a + " Y/N"] = df[a + 'Y',"sum"].astype("str") + "/" + df[a + 'N',"sum"].astype("str")
    del df[a + 'Y']
    del df[a + 'N']
    
merge(r,"Survival")
merge(r,"Went ICU")
merge(r,"Intubation")
r

filename    age COVID-19  \
                                            count   mean      sum   
region   sub-region                                                 
Africa   Northern Africa                        5  40.00        0   
Americas Latin America and the Caribbean        3  44.67        1   
         Northern America                      24  51.04       14   
Asia     Eastern Asia                          48  54.32       45   
         South-eastern Asia                     9  46.67        8   
         Southern Asia                         14  48.86       13   
         Western Asia                          10  42.40        5   
Europe   Eastern Europe                         3  65.00        0   
         Northern Europe                       33  57.83       22   
         Southern Europe                      100  57.49       68   
         Western Europe                        34  56.40       30   
Oceania  Australia and New Zealand             39  50.65        1   
Unknown  Unknown                               30  52.62       18   

                                         RT-PCR Positive Male Ratio  \
                                                     sum       mean   
region   sub-region                                                   
Africa   Northern Africa                               0       0.00   
Americas Latin America and the Caribbean               1       0.67   
         Northern America                             13       0.62   
Asia     Eastern Asia                                 35       0.31   
         South-eastern Asia                            8       0.56   
         Southern Asia                                13       0.71   
         Western Asia                                  2       0.80   
Europe   Eastern Europe                                0       0.67   
         Northern Europe                              18       0.55   
         Southern Europe                              55       0.61   
         Western Europe                                4       0.59   
Oceania  Australia and New Zealand                     2       0.54   
Unknown  Unknown                                       0       0.63   

                                         Survival Y/N Went ICU Y/N  \
                                                                     
region   sub-region                                                  
Africa   Northern Africa                          0/1          0/0   
Americas Latin America and the Caribbean          1/0          0/1   
         Northern America                         6/4          8/2   
Asia     Eastern Asia                            16/3          3/5   
         South-eastern Asia                       4/1          1/3   
         Southern Asia                            4/0          3/0   
         Western Asia                             2/0          2/0   
Europe   Eastern Europe                           0/0          0/0   
         Northern Europe                          3/3          8/0   
         Southern Europe                         28/2         18/9   
         Western Europe                           8/0         7/23   
Oceania  Australia and New Zealand                4/0          1/1   
Unknown  Unknown                                  3/2          5/0   

                                         Intubation Y/N  
                                                         
region   sub-region                                      
Africa   Northern Africa                            0/0  
Americas Latin America and the Caribbean            0/1  
         Northern America                           7/5  
Asia     Eastern Asia                               5/6  
         South-eastern Asia                         1/3  
         Southern Asia                              3/0  
         Western Asia                               1/0  
Europe   Eastern Europe                             0/0  
         Northern Europe                

In [28]:
r = dmerge.csv.groupby(["view"]).agg({
                                                         'filename':["count"],
                                                         'age':['mean'],
                                                         'SurvivalY':['sum'],
                                                         'SurvivalN':['sum'],
                                                         'Went ICUY':['sum'],
                                                         'Went ICUN':['sum'],
                                                         'IntubationY':['sum'],
                                                         'IntubationN':['sum'],
                                                         'COVID-19':['sum'],
                                                         'RT-PCR Positive':["sum"],
                                                         'Male Ratio':['mean'],
                                                        }).round(2)

def merge(df,a):
    df[a + " Y/N"] = df[a + 'Y',"sum"].astype("str") + "/" + df[a + 'N',"sum"].astype("str")
    del df[a + 'Y']
    del df[a + 'N']
    
merge(r,"Survival")
merge(r,"Went ICU")
merge(r,"Intubation")
#r["Survival Y/N"] = r['SurvivalY',"sum"].astype("str") + "/" + r['SurvivalN',"sum"].astype("str")

In [29]:
r

,filename,age,COVID-19,RT-PCR Positive,Male Ratio,Survival Y/N,Went ICU Y/N,Intubation Y/N
,count,mean,sum,sum,mean,,,
view,,,,,,,,
AP,203,57.52,146,114,0.54,37/13,62/7,53/18
PA,332,51.64,196,116,0.55,105/19,53/67,33/55


In [30]:
#specify task
task = "survival"

In [31]:
if task=="geographic-extent":
    print(task)
    target_str = "Geographic Extent"
    labels_mask = ~np.isnan(dmerge.csv.geographic_mean.astype(float))
    labels = dmerge.csv.geographic_mean[labels_mask]
    method = "linear"

if task=="opacity":
    print(task)
    target_str = "Opacity"
    labels_mask = ~np.isnan(dmerge.csv.opacity_mean.astype(float))
    labels = dmerge.csv.opacity_mean[labels_mask]
    method = "linear"

In [32]:
if task=="covid19":
    print(task)
    target_str = "COVID-19"
    labels_mask = ~dmerge.csv.finding.str.contains("todo") & \
                  (dmerge.csv.offset.astype("float") > 0) & \
                  (dmerge.csv.offset.astype("float") < 8) 
    labels = dmerge.csv.finding.str.contains("COVID-19")[labels_mask]
    method = "logistic"

In [33]:
if task=="viral-bacterial":
    print(task)
    target_str = "Viral or Bacterial"
    labels_mask = ((dmerge.csv.gt_Bacterial == 1) | (dmerge.csv.gt_Viral == 1))& \
                  (dmerge.csv.offset.astype("float") > 0) & \
                  (dmerge.csv.offset.astype("float") < 8) 
    labels = (dmerge.csv.gt_Viral == 1)[labels_mask]
    method = "logistic"

In [34]:
if task=="survival":
    print(task)
    target_str = "Survival"
    labels_mask = dmerge.csv.survival.isin(["Y","N"]) & \
                  ~(dmerge.csv.intubation_present == "Y") & \
                  (dmerge.csv.offset.astype("float") > 0) & \
                  (dmerge.csv.offset.astype("float") < 8)
    labels = dmerge.csv.survival[labels_mask] == "Y"
    method = "logistic"

survival


In [35]:
if task=="icu-stay":
    print(task)
    target_str = "ICU Stay"
    labels_mask = dmerge.csv.went_icu.isin(["Y","N"]) & \
                  ~(dmerge.csv.in_icu == "Y") & \
                  ~(dmerge.csv.intubation_present == "Y") & \
                  (dmerge.csv.offset.astype("float") > 0) & \
                  (dmerge.csv.offset.astype("float") < 8)
    labels = (dmerge.csv.went_icu[labels_mask] == "Y")[labels_mask]
    method = "logistic"

In [36]:
if task=="intubated":
    print(task)
    target_str = "Intubated"
    labels_mask = dmerge.csv.intubated.isin(["Y","N"]) & \
                  ~(dmerge.csv.intubation_present == "Y") & \
                  (dmerge.csv.offset.astype("float") > 0) & \
                  (dmerge.csv.offset.astype("float") < 8)
    labels = (dmerge.csv.intubated[labels_mask] == "Y")[labels_mask]
    method = "logistic"

In [37]:
labels_mask.sum()

np.int64(55)

In [38]:
dmerge.csv.region[labels_mask].unique()

<StringArray>
['Asia', 'Americas', 'Europe', 'Oceania', 'Unknown']
Length: 5, dtype: str

In [39]:
dmerge.csv["sub-region"][labels_mask].unique()

<StringArray>
[             'South-eastern Asia',                'Northern America',
                    'Eastern Asia',                 'Southern Europe',
       'Australia and New Zealand',                 'Northern Europe',
                         'Unknown',                  'Western Europe',
 'Latin America and the Caribbean',                   'Southern Asia']
Length: 10, dtype: str

In [40]:
groupby = "region"

In [41]:
test_regions = []
if method == "logistic":
    for region in dmerge.csv[groupby][labels_mask].dropna().unique():
        counts = labels[labels_mask][dmerge.csv[groupby][labels_mask] == region].value_counts()
        print(region, dict(counts))
        if len(counts) > 1 and counts.min() > 0:
            test_regions.append(region)        
if method == "linear":
    for region in dmerge.csv[groupby][labels_mask].dropna().unique():
        if (dmerge.csv[groupby][labels_mask] == region).sum() > 2:
            test_regions.append(region)    

if "Unknown" in test_regions:
    test_regions.remove("Unknown")
test_regions

Asia {True: np.int64(20), False: np.int64(3)}
Americas {True: np.int64(5), False: np.int64(1)}
Europe {True: np.int64(20), False: np.int64(3)}
Oceania {True: np.int64(1)}
Unknown {True: np.int64(2)}


['Asia', 'Americas', 'Europe']

In [42]:
results = pd.DataFrame()

In [43]:
data = d_feats[labels_mask]
for test_region in test_regions:
    results = pd.concat([results, pd.DataFrame([utils.evaluate(data, labels, "Intermediate features", plot=False, 
                                      regions = dmerge.csv[groupby][labels_mask],
                                      test_region=test_region,
                                      target_str=target_str, 
                                      method=method)])], ignore_index=True)
    

In [44]:
data = np.zeros((data.shape[0],1))
for test_region in test_regions:
    results = pd.concat([results, pd.DataFrame([utils.evaluate(data, labels, "Baseline prevalence", plot=False, 
                                      regions = dmerge.csv[groupby][labels_mask],
                                      test_region=test_region,
                                      target_str=target_str, 
                                      method=method)])], ignore_index=True)

In [45]:
data = d_preds[labels_mask]
for test_region in test_regions:
    results = pd.concat([results, pd.DataFrame([utils.evaluate(data, labels, "18 outputs", plot=False, 
                                      regions = dmerge.csv[groupby][labels_mask],
                                      test_region=test_region,
                                      target_str=target_str, 
                                      method=method)])], ignore_index=True)

In [46]:
data = d_preds[labels_mask]
data = data[:,[xrv.datasets.default_pathologies.index("Lung Opacity"),
               xrv.datasets.default_pathologies.index("Pneumonia"),
               xrv.datasets.default_pathologies.index("Infiltration"),
               xrv.datasets.default_pathologies.index("Consolidation")]]

for test_region in test_regions:
    results = pd.concat([results, pd.DataFrame([utils.evaluate(data, labels, "4 outputs", plot=False, 
                                      regions = dmerge.csv[groupby][labels_mask],
                                      test_region=test_region,
                                      target_str=target_str, 
                                      method=method)])], ignore_index=True)

In [47]:
data = d_preds[labels_mask]
data = data[:,[xrv.datasets.default_pathologies.index("Lung Opacity")]]
for test_region in test_regions:
    results = pd.concat([results, pd.DataFrame([utils.evaluate(data, labels, "lung opacity output", plot=False, 
                                      regions = dmerge.csv[groupby][labels_mask],
                                      test_region=test_region,
                                      target_str=target_str, 
                                      method=method)])], ignore_index=True)

In [48]:
results

,name,test_region,AUROC,AUPRC,# test samples,# params,method
0,Intermediate features,Asia,0.500000,0.869565,"{True: 20, False: 3}",1024+1,logistic
1,Intermediate features,Americas,0.500000,0.833333,"{False: 1, True: 5}",1024+1,logistic
2,Intermediate features,Europe,0.500000,0.869565,"{True: 20, False: 3}",1024+1,logistic
3,Baseline prevalence,Asia,0.500000,0.869565,"{True: 20, False: 3}",1+1,logistic
4,Baseline prevalence,Americas,0.500000,0.833333,"{False: 1, True: 5}",1+1,logistic
5,Baseline prevalence,Europe,0.500000,0.869565,"{True: 20, False: 3}",1+1,logistic
6,18 outputs,Asia,0.450000,0.858385,"{True: 20, False: 3}",18+1,logistic
7,18 outputs,Americas,0.500000,0.833333,"{False: 1, True: 5}",18+1,logistic
8,18 outputs,Europe,0.450000,0.858385,"{True: 20, False: 3}",18+1,logistic
9,4 outputs,Asia,0.500000,0.869565,"{True: 20, False: 3}",4+1,logistic


In [49]:
if method=="linear":
    a = results.groupby(["name", "method"]).agg({
                                 'Correlation':['mean','std'],
                                 'R^2':['mean','std'],
                                 'MAE':['mean','std'],
                                # 'MSE':['mean','std']}
                                }).round(2)
if method=="logistic":
    a = results.groupby(["name", "method"]).agg({
                                 'AUROC':['mean','std'],
                                 'AUPRC':['mean','std']}).round(2)
a

AUROC       AUPRC      
                                mean   std  mean   std
name                  method                          
18 outputs            logistic  0.47  0.03  0.85  0.01
4 outputs             logistic  0.54  0.07  0.87  0.03
Baseline prevalence   logistic  0.50  0.00  0.86  0.02
Intermediate features logistic  0.50  0.00  0.86  0.02
lung opacity output   logistic  0.55  0.08  0.87  0.03

In [50]:
results[" "] = ""

In [51]:
if method=="linear":
    a = results.groupby([" "," ","name", "# params"]).agg({
                             'Correlation':['mean','std'],
                             'R^2':['mean','std'],
                             'MAE':['mean','std']}).round(2).sort_values(("MAE", "mean"))
if method=="logistic":
    a = results.groupby([" "," ","name", "# params"]).agg({
                                 'AUROC':['mean','std'],
                                 'AUPRC':['mean','std']}).round(2).sort_values(("AUROC", "mean"), ascending=False)
    

for c in a.columns.get_level_values(0).unique():
    a[c+"_"] =  a[(c, "mean")].apply("{0:0.2f}".format) + "$\pm$" + a[(c, "std")].apply("{0:0.2f}".format)
    del a[c]
    a[c] = a[c+"_"]
    del a[c+"_"]


In [52]:
a

AUROC          AUPRC
                                                                
    name                  # params                              
    lung opacity output   1+1       0.55$\pm$0.08  0.87$\pm$0.03
    4 outputs             4+1       0.54$\pm$0.07  0.87$\pm$0.03
    Baseline prevalence   1+1       0.50$\pm$0.00  0.86$\pm$0.02
    Intermediate features 1024+1    0.50$\pm$0.00  0.86$\pm$0.02
    18 outputs            18+1      0.47$\pm$0.03  0.85$\pm$0.01

In [53]:
print(", ".join(sorted(test_regions)))

Americas, Asia, Europe


In [54]:
", ".join(sorted(results.test_region.unique()))

'Americas, Asia, Europe'

In [55]:
print(target_str + "\\\\N=" + str(labels_mask.sum()))

Survival\\N=55


In [56]:
print(target_str + "\\\\" + "0<offset<8 days" + 
                   "\\\\" + "True=" + str(labels[labels_mask].value_counts()[True]) + 
                   "\\\\" + "False=" + str(labels[labels_mask].value_counts()[False]))

Survival\\0<offset<8 days\\True=48\\False=7


## Simple extension: region-aware classifier benchmark

This appended section keeps the original notebook logic intact and adds a lightweight downstream classifier using the extracted image features. It compares a few simple models with leave-one-region-out evaluation to see whether a more flexible model improves survival prediction.

In [ ]:
 from sklearn.base import clone
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
import numpy as np

valid_mask = labels_mask & (dmerge.csv[groupby].fillna("Unknown").astype(str) != "Unknown")
X = d_feats[valid_mask.to_numpy()]
y = dmerge.csv.loc[valid_mask, "survival"].eq("Y").astype(int).to_numpy()
groups = dmerge.csv.loc[valid_mask, groupby].astype(str).to_numpy()

models = {
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, class_weight="balanced", C=1.0)),
    "svc_rbf": make_pipeline(StandardScaler(), SVC(C=3.0, kernel="rbf", probability=True, class_weight="balanced", random_state=0)),
    "extra_trees": ExtraTreesClassifier(n_estimators=500, random_state=0, class_weight="balanced"),
    "random_forest": RandomForestClassifier(n_estimators=500, random_state=0, class_weight="balanced")
}

logo = LeaveOneGroupOut()
results = {}
for name, model in models.items():
    aucs = []
    auprcs = []
    for train_idx, test_idx in logo.split(X, y, groups):
        if len(np.unique(y[test_idx])) < 2:
            continue
        model_fit = clone(model)
        model_fit.fit(X[train_idx], y[train_idx])
        proba = model_fit.predict_proba(X[test_idx])[:, 1]
        aucs.append(roc_auc_score(y[test_idx], proba))
        auprcs.append(average_precision_score(y[test_idx], proba))
    results[name] = {"AUROC": float(np.mean(aucs)) if aucs else float("nan"), "AUPRC": float(np.mean(auprcs)) if auprcs else float("nan")}

best_model = max(results.items(), key=lambda kv: kv[1]["AUROC"])
print("Leave-one-region-out AUROC/AUPRC by model:")
for k, v in sorted(results.items(), key=lambda kv: kv[1]["AUROC"], reverse=True):
    print(f"{k}: AUROC={v["AUROC"]:.3f}, AUPRC={v["AUPRC"]:.3f}")
print("\nBest model:", best_model[0], "with AUROC", round(best_model[1]["AUROC"], 3), "and AUPRC", round(best_model[1]["AUPRC"], 3))


Leave-one-region-out AUROC/AUPRC by model:
extra_trees: AUROC=0.697, AUPRC=0.943
random_forest: AUROC=0.683, AUPRC=0.935
logreg: AUROC=0.678, AUPRC=0.936
svc_rbf: AUROC=0.564, AUPRC=0.895

Best model: extra_trees with AUROC 0.697 and AUPRC 0.943
